# 06 Baselines

This notebook scores simple secret scanners on the same test repositories as the models in `04_models.ipynb`, so the models have something to be compared with.

The first baseline is a regex only scanner of the kind many pre-commit tools use: it flags a line when a secret keyword sits next to an assignment, or when the line matches a known key format.

As in the other notebooks, outputs show counts and scores only, never `LineText` or `Value`.

## Setup

We load the cleaned rows with `load_data` from `src/features.py` and keep the rows from the test repositories in `repo_split.csv`, the split made in `03_preprocessing.ipynb`.

In [1]:
import re
import sys
from pathlib import Path

import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

# The notebook runs from the notebooks folder so src and data sit one level up
sys.path.insert(0, str(Path("..") / "src"))
from features import load_data

PROCESSED = Path("..") / "data" / "processed"
pd.set_option("display.width", 200)

df = load_data()
split = pd.read_csv(PROCESSED / "repo_split.csv", dtype=str)
test_repos = set(split.loc[split["split"] == "test", "RepoId"])
test = df[df["RepoId"].isin(test_repos)].reset_index(drop=True)

print("test repositories:", len(test_repos))
print("test rows:", len(test))
print("label 1 share:", round(test["label"].mean() * 100, 2), "percent")
print(test["GroundTruth"].value_counts().reindex(["T", "F", "X"]).to_string())

test repositories: 67
test rows: 13235
label 1 share: 23.26 percent
GroundTruth
T    3078
F    8786
X    1371


## Regex baseline

The scanner looks at the line text only. A line is flagged when either kind of pattern matches:

- **Keyword next to an assignment**: a name containing password, secret, token, api key, access key, private key, auth or credential, followed by `=`, `:` or `:=`.
- **Known key formats**: a private key header, an AWS access key id, a GitHub token, a Slack token, a Google API key, a JSON Web Token, a Bearer token and a URL with a user and password in it.

These are written by hand and are not tuned on any data, so the baseline has nothing fitted on train or test.

In [2]:
KEYWORD_ASSIGN = re.compile(
    r"(?i)[a-z0-9_.\-]*(?:passw(?:or)?d|pwd|secret|token|api[_\-]?key|access[_\-]?key|private[_\-]?key|auth|credential)"
    r"[a-z0-9_.\-]*[\"']?\s*(?::=|=|:)"
)
KNOWN_FORMATS = {
    "private key header": r"-----BEGIN [A-Z ]*PRIVATE KEY",
    "aws access key id": r"\b(?:AKIA|ASIA)[0-9A-Z]{16}\b",
    "github token": r"\bgh[pousr]_[A-Za-z0-9]{36,}",
    "slack token": r"\bxox[abposr]-[A-Za-z0-9-]{10,}",
    "google api key": r"\bAIza[0-9A-Za-z_\-]{35}",
    "json web token": r"\beyJ[A-Za-z0-9_\-]{5,}\.[A-Za-z0-9_\-]{5,}\.[A-Za-z0-9_\-]{5,}",
    "bearer token": r"\b(?i:bearer)\s+[A-Za-z0-9._~+/\-]{16,}",
    "url with password": r"[a-z][a-z0-9+.\-]*://[^\s:/@]+:[^\s/@]+@",
}
KNOWN = re.compile("|".join(f"(?:{p})" for p in KNOWN_FORMATS.values()))

text = test["LineText"]
test["regex_keyword"] = text.map(lambda line: bool(KEYWORD_ASSIGN.search(line))).astype(int)
test["regex_format"] = text.map(lambda line: bool(KNOWN.search(line))).astype(int)
test["regex_flag"] = (test["regex_keyword"] | test["regex_format"]).astype(int)

for name, pattern in KNOWN_FORMATS.items():
    hits = text.str.contains(pattern, regex=True)
    print(f"{name}: {int(hits.sum())} rows, {round(test.loc[hits, 'label'].mean() * 100, 1) if hits.any() else '-'} percent real")

private key header: 134 rows, 82.8 percent real
aws access key id: 1 rows, 0.0 percent real
github token: 0 rows, - percent real
slack token: 2 rows, 100.0 percent real
google api key: 4 rows, 75.0 percent real


json web token: 52 rows, 96.2 percent real
bearer token: 20 rows, 100.0 percent real
url with password: 255 rows, 42.0 percent real


## Scores

Precision is the share of flagged lines that are real secrets, recall is the share of real secrets that get flagged, and F1 combines the two. We score the keyword rule and the format rule on their own and together.

In [3]:
def scores(name, flag):
    # Scores one baseline against the label on the test rows
    return {
        "baseline": name,
        "flagged rows": int(flag.sum()),
        "precision": round(precision_score(test["label"], flag, zero_division=0), 4),
        "recall": round(recall_score(test["label"], flag, zero_division=0), 4),
        "f1": round(f1_score(test["label"], flag, zero_division=0), 4),
    }


results = pd.DataFrame([
    scores("regex keyword next to assignment", test["regex_keyword"]),
    scores("regex known key formats", test["regex_format"]),
    scores("regex both", test["regex_flag"]),
    scores("flag every line", pd.Series(1, index=test.index)),
])
print(results.to_string(index=False))

                        baseline  flagged rows  precision  recall     f1
regex keyword next to assignment          6610     0.2434  0.5227 0.3322
         regex known key formats           464     0.6228  0.0939 0.1632
                      regex both          7024     0.2651  0.6049 0.3686
                 flag every line         13235     0.2326  1.0000 0.3774

## What the regex baseline flags by GroundTruth

The share of rows flagged in each GroundTruth group shows whether the scanner can tell F and X apart from real secrets at all.

In [4]:
by_gt = test.groupby("GroundTruth")[["regex_keyword", "regex_format", "regex_flag"]].mean().mul(100).round(1)
print(by_gt.reindex(["T", "F", "X"]).to_string())

             regex_keyword  regex_format  regex_flag
GroundTruth                                         
T                     52.3           9.4        60.5
F                     42.7           1.2        43.9
X                     90.9           4.9        95.5
